In [1]:
import pandas as pd
import json

# загружаем наш json
# ставим две точки (..), так как ноутбук лежит в папке notebooks, а данные на уровень выше
with open('../data/raw/steam_raw.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# превращаем словарь в датафрейм (таблицу)
df = pd.DataFrame.from_dict(raw_data, orient='index')

# смотрим, что получилось
display(df.head())
print(f"Размер сырой таблицы: {df.shape}")

,type,name,steam_appid,required_age,is_free,controller_support,dlc,detailed_description,about_the_game,short_description,...,recommendations,achievements,release_date,support_info,background,background_raw,content_descriptors,ratings,legal_notice,demos
413150,game,Stardew Valley,413150,0,False,full,[440820],Stardew Valley is an open-ended country-life R...,Stardew Valley is an open-ended country-life R...,You've inherited your grandfather's old farm p...,...,{'total': 866432},"{'total': 49, 'highlighted': [{'icon': 'dac82a...","{'coming_soon': False, 'date': '26 Feb, 2016'}","{'url': '', 'email': 'support@stardewvalley.net'}",https://store.akamai.steamstatic.com/images/st...,https://store.akamai.steamstatic.com/images/st...,"{'ids': [], 'notes': None}","{'esrb': {'rating': 'e10', 'descriptors': 'Fan...",NaN,NaN
367520,game,Hollow Knight,367520,0,False,full,"[598190, 916000]","<h2 class=""bb_tag"" >Hollow Knight Expands with...","<h2 class=""bb_tag"" >Hollow Knight Expands with...",Forge your own path in Hollow Knight! An epic ...,...,{'total': 485896},"{'total': 63, 'highlighted': [{'icon': '6d15e6...","{'coming_soon': False, 'date': '24 Feb, 2017'}","{'url': '', 'email': 'info@teamcherry.com.au'}",https://store.akamai.steamstatic.com/images/st...,https://store.akamai.steamstatic.com/images/st...,"{'ids': [], 'notes': None}","{'esrb': {'rating': 'e10', 'descriptors': 'Fan...",Hollow Knight is © Copyright Team Cherry 2019,NaN
105600,game,Terraria,105600,0,False,full,"[409210, 1323320]","Dig, Fight, Explore, Build: The very world is...","Dig, Fight, Explore, Build: The very world is...","Dig, fight, explore, build! Nothing is impossi...",...,{'total': 1215208},"{'total': 137, 'highlighted': [{'icon': '0fbb3...","{'coming_soon': False, 'date': '16 May, 2011'}",{'url': 'https://forums.terraria.org/index.php...,https://store.akamai.steamstatic.com/images/st...,https://shared.akamai.steamstatic.com/store_it...,"{'ids': [], 'notes': None}","{'dejus': {'rating_generated': '1', 'rating': ...",NaN,NaN
1145360,game,Hades,1145360,0,False,full,[1206340],"<span class=""bb_img_ctn""><video class=""bb_img""...","<span class=""bb_img_ctn""><video class=""bb_img""...",Defy the god of the dead as you hack and slash...,...,{'total': 281036},"{'total': 49, 'highlighted': [{'icon': '062a2b...","{'coming_soon': False, 'date': '17 Sep, 2020'}",{'url': 'https://www.supergiantgames.com/conta...,https://store.akamai.steamstatic.com/images/st...,https://shared.akamai.steamstatic.com/store_it...,"{'ids': [], 'notes': None}","{'dejus': {'rating': '14', 'descriptors': 'Vio...","© Supergiant Games, LLC 2020. All rights reser...",NaN
250900,game,The Binding of Isaac: Rebirth,250900,0,False,full,"[3353470, 1426300, 401920, 570660, 322660]","<h1>New DLC Available</h1><p><p class=""bb_para...",When Isaac’s mother starts hearing the voice o...,The Binding of Isaac: Rebirth is a randomly ge...,...,{'total': 356917},"{'total': 641, 'highlighted': [{'icon': 'a36d7...","{'coming_soon': False, 'date': '4 Nov, 2014'}","{'url': 'http://www.nicalis.com', 'email': 'is...",https://store.akamai.steamstatic.com/images/st...,https://store.akamai.steamstatic.com/images/st...,"{'ids': [5], 'notes': None}","{'esrb': {'required_age': '17', 'use_age_gate'...","© 2014 Nicalis, Inc/Edmund McMillen",NaN


Размер сырой таблицы: (31, 40)


In [2]:
# смотрим, какую свалку колонок нам отдал стим
print("исходные колонки:", df.columns.tolist())

# шаг 1: выкидываем весь текстовый и визуальный мусор
# модели машинного обучения не умеют смотреть на картинки и читать html-теги
trash_cols = [
    'detailed_description', 'about_the_game', 'short_description',
    'header_image', 'website', 'pc_requirements', 'mac_requirements', 
    'linux_requirements', 'legal_notice', 'reviews', 'support_info'
]

# errors='ignore' нужен, чтобы пандас не падал, если какой-то колонки вдруг нет
df_clean = df.drop(columns=trash_cols, errors='ignore')

# шаг 2: вытаскиваем нормальную цену
# в стиме цена лежит в словаре price_overview, а у бесплатных игр там вообще NaN
def extract_price(price_dict):
    # если это пустота (float/NaN) или игра бесплатная
    if pd.isna(price_dict) or not isinstance(price_dict, dict):
        return 0.0
    # стим хранит цены в копейках/центах, так что делим на 100
    # безопасно достаем значение через .get(), чтобы не словить KeyError
    return price_dict.get('final', 0) / 100

# применяем нашу функцию ко всей колонке
df_clean['price_clean'] = df_clean['price_overview'].apply(extract_price)

# шаг 3: вытаскиваем оценку метакритика (это будет наша целевая переменная для предсказаний)
def extract_meta(meta_dict):
    if pd.isna(meta_dict) or not isinstance(meta_dict, dict):
        return None
    return meta_dict.get('score')

df_clean['metacritic_score'] = df_clean['metacritic'].apply(extract_meta)

# смотрим на результат (только самые важные колонки)
display(df_clean[['name', 'is_free', 'price_clean', 'metacritic_score']].head(10))

исходные колонки: ['type', 'name', 'steam_appid', 'required_age', 'is_free', 'controller_support', 'dlc', 'detailed_description', 'about_the_game', 'short_description', 'supported_languages', 'reviews', 'header_image', 'capsule_image', 'capsule_imagev5', 'website', 'pc_requirements', 'mac_requirements', 'linux_requirements', 'developers', 'publishers', 'price_overview', 'packages', 'package_groups', 'platforms', 'metacritic', 'categories', 'genres', 'screenshots', 'movies', 'recommendations', 'achievements', 'release_date', 'support_info', 'background', 'background_raw', 'content_descriptors', 'ratings', 'legal_notice', 'demos']


,name,is_free,price_clean,metacritic_score
413150,Stardew Valley,False,299.00,89.0
367520,Hollow Knight,False,12.79,87.0
105600,Terraria,False,8.50,83.0
1145360,Hades,False,20.99,93.0
250900,The Binding of Isaac: Rebirth,False,10.99,NaN
646570,Slay the Spire,False,19.99,89.0
892970,Valheim,False,15.49,NaN
730,Counter-Strike 2,True,0.00,NaN
570,Dota 2,True,0.00,90.0
292030,The Witcher 3: Wild Hunt,False,29.99,93.0


In [3]:
# шаг 4: вытаскиваем пользовательские рекомендации (наша новая целевая переменная)
def extract_recs(rec_dict):
    if pd.isna(rec_dict) or not isinstance(rec_dict, dict):
        return 0
    return rec_dict.get('total', 0)

df_clean['recommendations_total'] = df_clean['recommendations'].apply(extract_recs)

# шаг 5: распаковываем жанры. сейчас это список словарей вида [{'id': '1', 'description': 'Action'}]
# превращаем это в нормальный список строк: ['Action', 'Indie']
def extract_genres(genres_list):
    if not isinstance(genres_list, list):
        return []
    # используем list comprehension (лиды любят когда джуны его знают)
    return [g.get('description') for g in genres_list if isinstance(g, dict)]

df_clean['genres_clean'] = df_clean['genres'].apply(extract_genres)

# шаг 6: достаем год выхода игры
def extract_year(date_dict):
    if pd.isna(date_dict) or not isinstance(date_dict, dict):
        return None
    date_str = date_dict.get('date', '')
    # обычно дата выглядит как "26 Feb, 2016", берем последние 4 символа
    if len(date_str) >= 4:
        return date_str[-4:]
    return None

df_clean['release_year'] = df_clean['release_date'].apply(extract_year)

# смотрим итоговый датасет, с которым уже можно делать машинное обучение
final_cols = ['name', 'price_clean', 'recommendations_total', 'genres_clean', 'release_year']
display(df_clean[final_cols].head(10))

,name,price_clean,recommendations_total,genres_clean,release_year
413150,Stardew Valley,299.00,866432,"[Indie, RPG, Simulation]",2016
367520,Hollow Knight,12.79,485896,"[Action, Adventure, Indie]",2017
105600,Terraria,8.50,1215208,"[Action, Adventure, Indie, RPG]",2011
1145360,Hades,20.99,281036,"[Action, Indie, RPG]",2020
250900,The Binding of Isaac: Rebirth,10.99,356917,[Action],2014
646570,Slay the Spire,19.99,188900,"[Indie, Strategy]",2019
892970,Valheim,15.49,452413,"[Action, Adventure, Indie, RPG, Early Access]",2021
730,Counter-Strike 2,0.00,5098157,"[Action, Free To Play]",2012
570,Dota 2,0.00,14371,"[Action, Strategy, Free To Play]",2013
292030,The Witcher 3: Wild Hunt,29.99,815756,[RPG],2015


In [4]:
# шаг 7: превращаем списки жанров в нули и единицы (One-Hot Encoding)
# склеиваем жанры через палку (|) и просим пандас сделать из них отдельные колонки
genres_dummies = df_clean['genres_clean'].str.join('|').str.get_dummies()

# шаг 8: собираем финальную таблицу
# склеиваем базовые колонки и новые колонки с жанрами
cols_to_keep = ['name', 'price_clean', 'recommendations_total', 'release_year']
df_ml = pd.concat([df_clean[cols_to_keep], genres_dummies], axis=1)

# шаг 9: финальная зачистка
# выкидываем игры без года релиза (модель сломается об NaN)
df_ml = df_ml.dropna(subset=['release_year'])

# переводим год из текста в нормальное число (int)
df_ml['release_year'] = df_ml['release_year'].astype(int)

# смотрим на наш шедевр
display(df_ml.head())
print(f"Размер таблицы для ML: {df_ml.shape}")

,name,price_clean,recommendations_total,release_year,Action,Adventure,Casual,Early Access,Free To Play,Indie,RPG,Simulation,Strategy
413150,Stardew Valley,299.00,866432,2016,0,0,0,0,0,1,1,1,0
367520,Hollow Knight,12.79,485896,2017,1,1,0,0,0,1,0,0,0
105600,Terraria,8.50,1215208,2011,1,1,0,0,0,1,1,0,0
1145360,Hades,20.99,281036,2020,1,0,0,0,0,1,1,0,0
250900,The Binding of Isaac: Rebirth,10.99,356917,2014,1,0,0,0,0,0,0,0,0


Размер таблицы для ML: (31, 13)


In [5]:
# шаг 10: размечаем хиты. допустим, игра "взлетела" (is_hit = 1), 
# если у нее больше 10000 рекомендаций. остальные - провал (0).
df_ml['is_hit'] = (df_ml['recommendations_total'] > 10000).astype(int)

# сохраняем готовую таблицу в папку processed (чистые данные)
# index=False нужен, чтобы пандас не сохранял номера строк как отдельную колонку
out_path = '../data/processed/steam_cleaned.csv'
df_ml.to_csv(out_path, index=False)

print(f"чистые данные сохранены в {out_path}! юпитер свою работу выполнил.")

чистые данные сохранены в ../data/processed/steam_cleaned.csv! юпитер свою работу выполнил.
